<a href="https://colab.research.google.com/github/michaelsteven1299/proyecto_michael-/blob/main/src/00_descargas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Universidad Libre - Seccional Cali<br>Facultad de Ingeniería - Diplomado en Ciencia de Datos<br>(ↄ) Diego Fernando Marin, 2024

# 00_descargas
Plantilla para el desarrollo del proyecto del diplomado de Ciencia de Datos, aplicando buenas prácticas.

---

Este cuaderno representa el primer paso crucial en nuestro proceso de ciencia de datos: la obtención y almacenamiento de datos crudos. El principio fundamental aquí es preservar los datos en su estado original, sin modificaciones, para garantizar la reproducibilidad del análisis y mantener una referencia histórica confiable.

**Propósito:** Establecer el punto de partida del análisis mediante la recopilación y almacenamiento de datos crudos de múltiples fuentes, manteniendo la integridad y trazabilidad de la información original.

**Tareas habituales:**
- Configuración de conexiones a bases de datos SQL y ejecución de consultas
- Implementación de web scraping para extracción de datos de páginas web
- Desarrollo de scripts RPA (Robotic Process Automation) para automatizar descargas
- Autenticación y descarga de datos desde APIs
- Verificación de integridad de archivos descargados
- Documentación de fuentes, timestamps y métodos de obtención
- Establecimiento de estructura de carpetas para datos crudos
- Implementación de control de versiones para datos cuando sea aplicable

# DESCARGA DE DATOS DE YAHOO FINANCE

In [6]:
from google.colab import drive
import os, pandas as pd, yfinance as yf

drive.mount('/content/drive')

RAW_PATH = "/content/drive/MyDrive/proyecto_oro/data/raw/"
os.makedirs(RAW_PATH, exist_ok=True)

tickers = {
    "GLD":      "oro_xauusd",
    "DX-Y.NYB": "dxy",
    "SLV":      "plata_xagusd",
    "CL=F":     "wti_crudo",
    "^TNX":     "bono_10y",
    "^VIX":     "vix",
    "EURUSD=X": "eur_usd",
    "^GVZ":     "gvz",
    "HG=F":     "cobre",
}

for ticker, nombre in tickers.items():
    try:
        df = yf.download(ticker, start="2021-01-01", end="2026-07-01",
                          auto_adjust=True, progress=False)

        if df.empty:
            print(f"⚠ Sin datos: {ticker}")
            continue

        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        df.to_csv(RAW_PATH + f"{nombre}.csv")
        print(f"✓ {nombre}.csv — {len(df)} filas")

    except Exception as e:
        print(f"X {ticker}: {e}")



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ oro_xauusd.csv — 1378 filas
✓ dxy.csv — 1380 filas
✓ plata_xagusd.csv — 1378 filas
✓ wti_crudo.csv — 1380 filas
✓ bono_10y.csv — 1378 filas
✓ vix.csv — 1379 filas
✓ eur_usd.csv — 1428 filas
✓ gvz.csv — 1378 filas
✓ cobre.csv — 1381 filas


In [2]:
!pip install fredapi -q

In [4]:
from fredapi import Fred

FRED_API_KEY = "c5c3cbde66ca205fc83e9e826f77a432"  # tu key real de FRED
fred = Fred(api_key=FRED_API_KEY)

series_fred = {
    "DFII10": "yield_real_10y",
    "T10YIE": "breakeven_inflacion",
    "T10Y2Y": "spread_curva_10y2y",
    "DGS2":   "bono_2y",
}

for serie_id, nombre in series_fred.items():
    try:
        data = fred.get_series(serie_id, observation_start="2021-01-01", observation_end="2026-07-01")
        df = data.to_frame(name=nombre)
        df.index.name = "Date"
        df.to_csv(RAW_PATH + f"{nombre}.csv")
        print(f"✓ {nombre}.csv — {len(df)} filas")
    except Exception as e:
        print(f"X {serie_id}: {e}")

✓ yield_real_10y.csv — 1434 filas
✓ breakeven_inflacion.csv — 1434 filas
✓ spread_curva_10y2y.csv — 1434 filas
✓ bono_2y.csv — 1434 filas
